### Simle GenAI Application with Langchain & OpenAI

### 1. Loading required library & environment variables

In [34]:
import os
from dotenv import load_dotenv

load_dotenv()

## Loading .env

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv("LANGCHAIN_PROJECT")

### 2. Data Ingestion

Flow or Steps to perform:
* Load the Data from website below,
* Convert it to Documents
* Split it to the Chunks
* Later pass it for embeddings & vector store

Scraping text from a [Trace an LLM application tutorial](https://docs.langchain.com/langsmith/observability-llm-tutorial#trace-an-llm-application-tutorial)

In [3]:
##Getting the documents from website

from langchain_community.document_loaders import WebBaseLoader

loader=WebBaseLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial#trace-an-llm-application-tutorial")
loader

In [6]:
## Viewing the loaded documents
data=loader.load()
print(data)

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial#trace-an-llm-application-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsTrace an LLM applicationPolly AI assistantBetaTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previewsQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLILangSmith MCP ServerView server logs for a traceBulk export trac

In [8]:
## Chunking the documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_spliter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
doc_chunks=text_spliter.split_documents(data)
doc_chunks

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial#trace-an-llm-application-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewQuickstartConceptsTrace an LLM applicationPolly AI assistantBetaTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previewsQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLILangSmith MCP ServerView server logs for a traceBulk export trac

In [9]:
## Calling embedding model from OpenAI

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7580dd7c7e00>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7580dd258830>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [11]:
## Calling FAISS vectorstore to store the document chunks and their embeddings
from langchain_community.vectorstores import FAISS

vectorstoredb=FAISS.from_documents(documents=doc_chunks,embedding=embeddings)
vectorstoredb

In [15]:
## Running similarity search and retriever on given query
query="To send traces to a specific project, use the"
query_result=vectorstoredb.similarity_search(query)
query_result[0].page_content

'To send traces to a specific project, use the LANGSMITH_PROJECT environment variable. If this is not set, LangSmith will create a default tracing project automatically on trace ingestion.\nYou may see these variables referenced as LANGCHAIN_* in other places. These are all equivalent, however the best practice is to use LANGSMITH_TRACING, LANGSMITH_API_KEY, LANGSMITH_PROJECT.The LANGSMITH_PROJECT flag is only supported in JS SDK versions >= 0.2.16, use LANGCHAIN_PROJECT instead if you are using an older version.\n\u200bTrace your LLM calls\nThe first thing you might want to trace is all your OpenAI calls. After all, this is where the LLM is actually being called, so it is the most important part! We’ve tried to make this as easy as possible with LangSmith by introducing a dead-simple OpenAI wrapper. All you have to do is modify your code to look something like:\nPythonTypeScriptCopyfrom openai import OpenAI\nfrom langsmith.wrappers import wrap_openai\nopenai_client = wrap_openai(OpenA

#### 3. Retrieval & Documents Chain 

In [33]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

## Calling LLM
llm = ChatOpenAI(model="gpt-4o-mini")

## Creating prompt template and document chain
prompt=ChatPromptTemplate.from_messages([
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    """
])

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_c

In [39]:
## Invoking the document chain on the retrieved documents and query
from langchain_core.documents import Document

document_chain.invoke({
    "input":"To send traces to a specific project, use the",
    "context":[Document(page_content="To send traces to a specific project, use the LANGSMITH_PROJECT environment variable. If this is not set, LangSmith will create a default tracing project automatically")]
})

'What should you do to send traces to a specific project in LangSmith?'

##### Retriever

In [40]:
## Input --> Retriever --> VectorstoreDB --> LLM --> Output

vectorstoredb

In [44]:
retriever=vectorstoredb.as_retriever()

from langchain_classic.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7580dd259fd0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
  

In [45]:
## Checking the final output after passing through the entire chain
retrieval_result=retrieval_chain.invoke({"input":"To send traces to a specific project, use the"})
retrieval_result['answer']

'What environment variable should you set to send traces to a specific project in LangSmith?'

In [49]:
retrieval_result['context'][0].page_content

'To send traces to a specific project, use the LANGSMITH_PROJECT environment variable. If this is not set, LangSmith will create a default tracing project automatically on trace ingestion.\nYou may see these variables referenced as LANGCHAIN_* in other places. These are all equivalent, however the best practice is to use LANGSMITH_TRACING, LANGSMITH_API_KEY, LANGSMITH_PROJECT.The LANGSMITH_PROJECT flag is only supported in JS SDK versions >= 0.2.16, use LANGCHAIN_PROJECT instead if you are using an older version.\n\u200bTrace your LLM calls\nThe first thing you might want to trace is all your OpenAI calls. After all, this is where the LLM is actually being called, so it is the most important part! We’ve tried to make this as easy as possible with LangSmith by introducing a dead-simple OpenAI wrapper. All you have to do is modify your code to look something like:\nPythonTypeScriptCopyfrom openai import OpenAI\nfrom langsmith.wrappers import wrap_openai\nopenai_client = wrap_openai(OpenA